# Notebook 05: LoRA Rank Ablation Study

### Research Question 6 (RQ6):
> *How does LoRA rank ($r$) affect predictive performance, trainable parameter count, GPU memory usage, and training time?*

### Experimental Setup
We systematically test four rank configurations:

$$r \in \{4, 8, 16, 32\}$$

Following standard practice, we set $\alpha = 2r$ so the scaling factor $\frac{\alpha}{r} = 2.0$ remains constant across all ranks.

### What we measure for each rank:
- Classification quality: Accuracy, Macro-F1, Weighted-F1
- Parameter count: Trainable parameters & percentage
- Resource efficiency: Peak GPU VRAM, Training duration, Adapter disk size

In [ ]:
# drive setup
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')

os.chdir('/content/drive/MyDrive/llm-project/lora-qlora-financial-sentiment')
sys.path.insert(0, os.getcwd())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'datasets', 'peft', 'accelerate', 'bitsandbytes',
    'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'pyyaml'],
    check=True)

for d in ['results/metrics', 'results/predictions', 'results/figures',
          'results/logs', 'adapters/lora', 'adapters/qlora']:
    os.makedirs(d, exist_ok=True)

print(f'Working directory: {os.getcwd()}')
print(f'src/ found: {os.path.exists("src")}')  # Must be True!
print('Setup complete! ✅')

Mounted at /content/drive
Working directory: /content/drive/.shortcut-targets-by-id/1fmKQoCL8RLydtggUMrG8oso1X-vif2bH/llm-project/lora-qlora-financial-sentiment
src/ found: True
Setup complete! ✅


In [ ]:
# Hardware check
import json
import torch


from src.model_utils import detect_hardware, print_hardware_summary
hw = detect_hardware()
print_hardware_summary(hw)

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

           HARDWARE & RUNTIME ENVIRONMENT SUMMARY           
CUDA Available:        True
GPU Name:              Tesla T4
GPU VRAM:              14.56 GB (14912.69 MB)
CUDA Version:          13.0
BF16 Supported:        True
PyTorch Version:       2.11.0+cu130
Transformers Version:  5.17.0
Compute Dtype:         bfloat16


In [ ]:
# Step 3: Load data
from src.data_utils import load_raw_dataset, create_stratified_split, prepare_hf_training_dataset, format_chat_prompt
from src.model_utils import load_tokenizer

tokenizer = load_tokenizer(MODEL_NAME)
df = load_raw_dataset('takala/financial_phrasebank', 'sentences_50agree')
train_df, val_df, test_df = create_stratified_split(df, 0.70, 0.15, 0.15, random_seed=42)

train_dataset = prepare_hf_training_dataset(train_df, tokenizer=tokenizer, max_length=256)
val_dataset = prepare_hf_training_dataset(val_df, tokenizer=tokenizer, max_length=256)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.58k [00:00<?, ?B/s]

financial_phrasebank.py:   0%|          | 0.00/6.04k [00:00<?, ?B/s]

Loading via `datasets` failed (Dataset scripts are no longer supported, but found financial_phrasebank.py). Using direct Hugging Face Hub download fallback...


In [ ]:
# Tokenize datasets if not already tokenized
if "text" in train_dataset.column_names:
    def tokenize_fn(examples):
        return tokenizer(examples["text"], truncation=True, max_length=256)

    cols_to_remove = [c for c in train_dataset.column_names if c not in ["input_ids", "attention_mask"]]
    train_dataset = train_dataset.map(tokenize_fn, batched=True, remove_columns=cols_to_remove)
    val_dataset = val_dataset.map(tokenize_fn, batched=True, remove_columns=cols_to_remove)
    print(f"Tokenized! Columns: {train_dataset.column_names}")
else:
    print(f"Already tokenized: {train_dataset.column_names}")

Map:   0%|          | 0/3392 [00:00<?, ? examples/s]

Map:   0%|          | 0/727 [00:00<?, ? examples/s]

Tokenized! Columns: ['input_ids', 'attention_mask']


In [5]:
# Step 4: Run Rank Ablation Loop

# --- TorchAO fix ---
import sys
if 'torchao' in sys.modules:
    sys.modules['torchao'].__version__ = '0.16.0'
for mod in list(sys.modules.values()):
    if hasattr(mod, 'is_torchao_available'):
        setattr(mod, 'is_torchao_available', lambda *a, **k: False)

from src.model_utils import (
    load_base_model, apply_lora_to_model, count_parameters,
    get_adapter_disk_size_mb, free_memory
)
from src.memory_utils import MemoryTracker
from src.metrics import parse_model_output, compute_classification_metrics
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from transformers.trainer_utils import get_last_checkpoint
import time, os, torch

# --- Tokenize dataset if not already tokenized ---
if "text" in train_dataset.column_names:
    def tokenize_fn(examples):
        return tokenizer(examples["text"], truncation=True, max_length=256)
    cols_to_remove = [c for c in train_dataset.column_names if c not in ["input_ids", "attention_mask"]]
    train_dataset = train_dataset.map(tokenize_fn, batched=True, remove_columns=cols_to_remove)
    val_dataset = val_dataset.map(tokenize_fn, batched=True, remove_columns=cols_to_remove)
    print(f"Tokenized! Columns: {train_dataset.column_names}")

RANKS = [4, 8, 16, 32]
ablation_results = []

ABLATION_CHECKPOINT_ROOT = (
    "/content/drive/MyDrive/llm-project/"
    "lora-qlora-financial-sentiment/checkpoints/ablation"
)
os.makedirs(ABLATION_CHECKPOINT_ROOT, exist_ok=True)

for r in RANKS:
    alpha = 2 * r
    print("=" * 60)
    print(f"Running Rank Ablation for r = {r} (alpha = {alpha})...")
    print("=" * 60)

    free_memory()

    # Load 4-bit base model
    base = load_base_model(MODEL_NAME, quantization=True)
    model = apply_lora_to_model(base, r=r, lora_alpha=alpha, is_qlora=True, gradient_checkpointing=True)
    param_stats = count_parameters(model)

    # Checkpoint directory for this rank
    checkpoint_dir = os.path.join(ABLATION_CHECKPOINT_ROOT, f"r{r}")
    os.makedirs(checkpoint_dir, exist_ok=True)
    last_checkpoint = get_last_checkpoint(checkpoint_dir)

    if last_checkpoint:
        print(f"Resuming r={r} from: {last_checkpoint}")
    else:
        print(f"Starting r={r} from scratch...")

    # Training arguments
    args = TrainingArguments(
        output_dir=checkpoint_dir,
        num_train_epochs=2,
        learning_rate=2e-4,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=50,
        weight_decay=0.01,
        logging_steps=30,
        eval_strategy="no",
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        bf16=hw["bf16_supported"],
        fp16=not hw["bf16_supported"] and hw["cuda_available"],
        gradient_checkpointing=True,
        report_to="none",
    )

    # Trainer (with data_collator to create labels from input_ids)
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    # Train / Resume
    with MemoryTracker(f"Ablation r={r}") as tracker:
        t0 = time.perf_counter()
        trainer.train(resume_from_checkpoint=last_checkpoint)
        train_time = time.perf_counter() - t0

    # Save adapter
    temp_dir = f"adapters/ablation_r{r}"
    os.makedirs(temp_dir, exist_ok=True)
    model.save_pretrained(temp_dir)
    adapter_size = get_adapter_disk_size_mb(temp_dir)

    # Evaluation on test set
    model.eval()
    device = next(model.parameters()).device
    preds = []
    for _, row in test_df.iterrows():
        prompt = format_chat_prompt(row["sentence"], tokenizer=tokenizer)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        preds.append(parse_model_output(gen))

    m = compute_classification_metrics(test_df["label_text"].tolist(), preds)

    # Store results
    ablation_results.append({
        "rank": r,
        "alpha": alpha,
        "accuracy": m["accuracy"],
        "macro_f1": m["macro_f1"],
        "weighted_f1": m["weighted_f1"],
        "trainable_parameters": param_stats["trainable_parameters"],
        "trainable_parameters_millions": param_stats["trainable_parameters_millions"],
        "trainable_percentage": param_stats["trainable_percentage"],
        "peak_vram_gb": tracker.peak_allocated_gb,
        "training_time_seconds": round(train_time, 2),
        "adapter_size_mb": adapter_size,
    })

    print(f"Done r={r}: Macro-F1={m['macro_f1']:.4f} | Params={param_stats['trainable_parameters_millions']}M | Time={train_time/60:.1f}m")

    # Free GPU memory before next rank
    del model, base, trainer
    free_memory()

Running Rank Ablation for r = 4 (alpha = 8)...


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Starting r=32 from scratch...


Step,Training Loss
30,3.157678
60,1.061996
90,0.983276
120,0.943422
150,0.975446
180,0.945281
210,0.944019
240,0.965855
270,0.926303
300,0.926684


Step,Training Loss
30,3.157678
60,1.061996
90,0.983276
120,0.943422
150,0.975446
180,0.945281
210,0.944019
240,0.965855
270,0.926303
300,0.926684


Done r=32: Macro-F1=0.8429 | Params=14.7456M | Time=127.9m


In [6]:
# Step 5: Save Ablation Results and Plot Curves
from src.visualization import plot_rank_ablation
import pandas as pd

with open('results/metrics/rank_ablation.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)

abl_df = pd.DataFrame(ablation_results)
print('=== Rank Ablation Summary ===')
display(abl_df[['rank', 'macro_f1', 'accuracy', 'trainable_parameters_millions', 'peak_vram_gb', 'training_time_seconds', 'adapter_size_mb']])

plot_rank_ablation(ablation_results, save_path='results/figures/rank_ablation.png')
print('\nSaved figure to results/figures/rank_ablation.png')

=== Rank Ablation Summary ===


,rank,macro_f1,accuracy,trainable_parameters_millions,peak_vram_gb,training_time_seconds,adapter_size_mb
0,4,0.8310,0.8391,1.8432,2.525,6.21,7.07
1,8,0.8335,0.8446,3.6864,2.555,6.46,14.10
2,16,0.8447,0.8528,7.3728,2.596,8.37,28.17
3,32,0.8429,0.8501,14.7456,3.800,7676.15,56.29



Saved figure to results/figures/rank_ablation.png


### Scientific Analysis of Rank Ablation

- **Capacity vs. Efficiency:** Increasing rank from 4 to 16 increases adapter capacity, but does Macro-F1 scale linearly, or do we see diminishing returns?
- **Parameter Growth:** Trainable parameters scale proportionally with $r$.
- In the next notebook, we synthesize all experimental outputs into our final **Efficiency Scorecard** and answer all research questions.